# Multimodal notebook

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
!pip install unsloth
import unsloth
from unsloth import FastVisionModel
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive

/tmp/ipykernel_13766/3528354911.py:7: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_100.json")


Mounted at /content/drive


In [ ]:
SYSTEM_PROMPT = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

SYSTEM_PROMPT_MULTIMODAL = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Atención: Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

SYSTEM_PROMPT_BRIEF_REASONING = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Atención: Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.
Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.

No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "razonamiento": "Aquí escribes una breve explicación de la respuesta elegida basándote en el texto.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}"""

CHAIN_OF_THOUGHT = """Eres un profesor experto en resolver exámenes de comprensión lectora en español.
Tu tarea es leer el texto, analizarlo y seleccionar la opción correcta para la pregunta planteada.

Para analizar el texto y responder a las preguntas, debes tener en cuenta estos aspectos:
1. LIMÍTATE AL CONTENIDO DEL TEXTO: Ignora cualquier conocimiento externo o sesgo personal. La validez de una opción depende exclusivamente de la información (explícita o implícita) contenida en el texto.
2. NO TE GUÍES POR LA EXTENSIÓN DE LAS RESPUESTAS: Una opción más detallada o larga no es necesariamente la correcta.
3. ANÁLISIS GLOBAL Y DESCARTE ARGUMENTADO: Evalúa el texto como una unidad (apóyate en marcadores del discurso, tiempos verbales y pronombres) y ten en cuenta que la respuesta correcta casi nunca estará escrita literalmente. Busca equivalencias de significado.
4. EVALÚA TODAS LAS ALTERNATIVAS: debes descartar las opciones incorrectas una a una, aportando argumentos claros de por qué el texto las contradice o no las respalda.

Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "razonamiento": "Aquí escribes todo el RAZONAMIENTO necesario para tomar la decisión de cuál es la opción correcta teniendo en cuenta las instrucciones anteriores.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}
"""

In [ ]:
def load_model_unsloth(model_name, max_seq_length=8192, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    FastVisionModel.for_inference(model)

    return model, tokenizer

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    return data, ground_truth

In [ ]:
def filter_questions(data, ground_truth):
    """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']

            for q in exercise['questions']:
                q_id = q['questionId']

                if q_id in ground_truth:
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": q['options'],
                        "real": ground_truth[q_id]
                    })
    return tareas

In [ ]:
from PIL import Image

def prepare_batch(batch, system_prompt, modo_salida):
    """
    Construye los mensajes en formato multimodal para un lote de tareas.
    Devuelve la lista de mensajes y una lista paralela con las imágenes cargadas.
    """
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []
        imagenes_tarea = []

        base_text = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n"
        user_content.append({"type": "text", "text": base_text})

        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})

            elif ruta_img:
                user_content.append({"type": "text", "text": f"{letra}) "})

                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)

                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        if modo_salida == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [ ]:
import torch

def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def show_results(stats, output_file):
    """Imprime por pantalla el resumen de la evaluación."""
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*50)
    print(f"RESULTADOS : {output_file}")
    print("="*50)
    if stats["errores_formato"] > 0:
        print(f"Errores de formato (JSON/Regex fallido): {stats['errores_formato']} de {stats['total']}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 50)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")

In [ ]:
def evaluate_model(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file="resultados.jsonl"
):
    """Función principal que orquesta todo el flujo."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    tareas = filter_questions(data, ground_truth)

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    if os.path.exists(output_file):
        os.remove(output_file)

    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
        textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

        batch_results = []

        for j, texto_bruto in enumerate(textos_generados):
            prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

            tarea_actual = batch[j]
            real = tarea_actual["real"]
            nivel = tarea_actual["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1
            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1


            if errors:
                stats["errores_formato"] += 1

            batch_results.append({
                "questionId": tarea_actual["id"],
                "nivel": nivel,
                "pregunta": tarea_actual["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "error_procesamiento_json": errors,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

        with open(output_file, 'a', encoding='utf-8') as f:
          for resultado in batch_results:
              linea_json = json.dumps(resultado, ensure_ascii=False)
              f.write(linea_json + '\n')

    show_results(stats, output_file)

In [ ]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen


def run_inference(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size_texto=4, # Batch size solo para textos
    output_file="resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    todas_las_tareas = filter_questions(data, ground_truth)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todas_las_tareas)
    if os.path.exists(output_file):
        os.remove(output_file)

    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

                tarea_actual = batch[j]
                real = tarea_actual["real"]
                nivel = tarea_actual["nivel"]
                es_correcto = (prediccion == real)

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": nivel,
                    "pregunta": tarea_actual["pregunta"],
                    "respuesta_real": real,
                    "prediccion_modelo": prediccion,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                    "estado": "CORRECTO" if es_correcto else "INCORRECTO"
                })

            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    linea_json = json.dumps(resultado, ensure_ascii=False)
                    f.write(linea_json + '\n')

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} tareas de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} tareas MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

In [ ]:
def calculate_metrics(input_file="resultados.jsonl"):
    """Lee las predicciones almacenadas y calcula las métricas finales."""

    if not os.path.exists(input_file):
        print(f"Error: No se ha encontrado el archivo {input_file}.")
        return

    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    with open(input_file, 'r', encoding='utf-8') as f:
        for linea in f:
            if not linea.strip():
                continue

            resultado = json.loads(linea)

            nivel = resultado["nivel"]
            es_correcto = (resultado["estado"] == "CORRECTO")
            error_json = resultado.get("error_procesamiento_json", False)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            if error_json:
                stats["errores_formato"] += 1

    show_results(stats, input_file)

## [Gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it) cuantizado


In [ ]:
gemma4_model, gemma4_tokenizer = load_model_unsloth("unsloth/gemma-4-E4B-it-unsloth-bnb-4bit")

==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results')

### zero-shot

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_zero_shot.json")

evaluate_model(
    model=gemma4_model,
    tokenizer=gemma4_tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=1,
    output_file=gemma4_results_path
)

Progreso: 100%|██████████| 135/135 [02:06<00:00,  1.07it/s]


RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_zero_shot.json
Accuracy Global: 86.67% (117/135)
--------------------------------------------------
Nivel A1: 91.55% (65/71)
Nivel A2: 78.26% (18/23)
Nivel B1: 80.95% (17/21)
Nivel B2: 85.00% (17/20)


### zero-shot con breve explicación

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_zero_shot_expl.json")

evaluate_model(
    model=gemma4_model,
    tokenizer=gemma4_tokenizer,
    system_prompt=SYSTEM_PROMPT_BRIEF_REASONING,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=1,
    output_file=gemma4_results_path
)

Progreso: 100%|██████████| 135/135 [41:03<00:00, 18.25s/it]


RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_zero_shot_expl.json
Accuracy Global: 87.41% (118/135)
--------------------------------------------------
Nivel A1: 92.96% (66/71)
Nivel A2: 73.91% (17/23)
Nivel B1: 90.48% (19/21)
Nivel B2: 80.00% (16/20)


### Chain of thought

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_cot.json")

evaluate_model(
    model=gemma4_model,
    tokenizer=gemma4_tokenizer,
    system_prompt=CHAIN_OF_THOUGHT_MULTIMODAL,
    modo_salida="json",
    max_new_tokens=1500,
    batch_size=1,
    output_file=gemma4_results_path
)

Progreso: 100%|██████████| 135/135 [2:11:27<00:00, 58.42s/it]


RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_cot.json
Accuracy Global: 88.15% (119/135)
--------------------------------------------------
Nivel A1: 91.55% (65/71)
Nivel A2: 86.96% (20/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 80.00% (16/20)


## [Gemma-4-E4B-it](https://huggingface.co/unsloth/gemma-4-E4B-it) sin cuantizar


In [ ]:
gemma4_model, gemma4_tokenizer = load_model_unsloth("unsloth/gemma-4-E4B-it")

==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results')

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_zero_shot_no_cuant.json")

run_inference(
    model=gemma4_model,
    tokenizer=gemma4_tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    batch_size_texto=16,
    output_file=gemma4_results_path
)

calculate_metrics(input_file=gemma4_results_path)


--- Procesando 128 tareas de SOLO TEXTO (Batch Size: 16) ---


Progreso Texto: 100%|██████████| 8/8 [00:34<00:00,  4.30s/it]



--- Procesando 7 tareas MULTIMODALES (Batch Size: 1) ---


Progreso Imágenes: 100%|██████████| 7/7 [00:40<00:00,  5.72s/it]


Resultados guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_zero_shot_no_cuant.json

RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/gemma4_zero_shot_no_cuant.json
Accuracy Global: 86.67% (117/135)
--------------------------------------------------
Nivel A1: 91.55% (65/71)
Nivel A2: 78.26% (18/23)
Nivel B1: 80.95% (17/21)
Nivel B2: 85.00% (17/20)


## [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) cuantizado

In [ ]:
qwen35_model, qwen35_tokenizer = load_model_unsloth("unsloth/Qwen3.5-9B")

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
qwen35_path = os.path.join(BASE_PATH, 'qwen35_results')

### zero-shot

In [ ]:
qwen35_results_path = os.path.join(qwen35_path, "qwen35_zero_shot.json")

evaluate_model(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=1,
    output_file=qwen35_results_path
)

Progreso: 100%|██████████| 135/135 [02:43<00:00,  1.21s/it]


RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_results/qwen35_zero_shot.json
Accuracy Global: 85.93% (116/135)
--------------------------------------------------
Nivel A1: 88.73% (63/71)
Nivel A2: 78.26% (18/23)
Nivel B1: 85.71% (18/21)
Nivel B2: 85.00% (17/20)


## [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) sin cuantizar

In [ ]:
qwen35_model, qwen35_tokenizer = load_model_unsloth("unsloth/Qwen3.5-9B", load_in_4bit=False)

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
qwen35_path = os.path.join(BASE_PATH, 'qwen35_results')

In [ ]:
qwen35_results_path = os.path.join(qwen35_path, "qwen35_zero_shot_no_cuant.json")

evaluate_model(
    model=qwen35_model,
    tokenizer=qwen35_tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    max_new_tokens=5,
    batch_size=1,
    output_file=qwen35_results_path
)

Progreso: 100%|██████████| 135/135 [02:17<00:00,  1.02s/it]


RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_results/qwen35_zero_shot_no_cuant.json
Accuracy Global: 91.85% (124/135)
--------------------------------------------------
Nivel A1: 94.37% (67/71)
Nivel A2: 91.30% (21/23)
Nivel B1: 90.48% (19/21)
Nivel B2: 85.00% (17/20)
